In [3]:
from secret_envs_wrapper import SecretEnv3
from collections import defaultdict, Counter
from tqdm import tqdm
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
import random
from DRL_algorithms.Dynamic_methods import (
    iterative_policy_evaluation_sparse,
    policy_iteration_sparse,
    value_iteration_sparse
)
from typing import Any, Dict, List, Tuple


In [2]:
# === 1. Initialiser l'environnement ===
env = SecretEnv3()

num_states = env.num_states()
num_actions = env.num_actions()
num_rewards = env.num_rewards()

print("Nombre d'états:", num_states)
print("Nombre d'actions:", num_actions)
print("Nombre de récompenses:", num_rewards)


Nombre d'états: 65536
Nombre d'actions: 3
Nombre de récompenses: 3


In [4]:

#  Instanciation de l’environnement
env = SecretEnv3()

#  Récupération des récompenses possibles
num_rewards = env.num_rewards()
R = [env.reward(i) for i in range(num_rewards)]

#  Structures pour le sampling
trans_counter = defaultdict(lambda: defaultdict(Counter))
valid_actions_dict = defaultdict(list)
S = set()

#  Paramètre à ajuster selon ton PC
nb_samples = 500_000

#  Exploration aléatoire pour construire le modèle empirique
env.reset()
for _ in tqdm(range(nb_samples)):
    s = env.state_id()
    S.add(s)

    if env.is_game_over():
        env.reset()
        continue

    a = random.choice(env.available_actions())
    env.step(a)
    s_p = env.state_id()
    S.add(s_p)

    r = env.score()
    done = env.is_game_over()

    trans_counter[s][a][(s_p, r, done)] += 1

    if a not in valid_actions_dict[s]:
        valid_actions_dict[s].append(a)

#  Construction du modèle empirique
model = defaultdict(lambda: defaultdict(list))
for s in trans_counter:
    for a in trans_counter[s]:
        total = sum(trans_counter[s][a].values())
        for (s_p, r, done), count in trans_counter[s][a].items():
            p = count / total
            model[s][a].append((p, s_p, r, done))


  0%|          | 0/500000 [00:00<?, ?it/s]

100%|██████████| 500000/500000 [00:06<00:00, 79785.06it/s]


In [5]:
# Politique uniforme pour l'évaluation
pi_uniform = {
    s: {a: 1.0 / len(valid_actions_dict[s]) for a in valid_actions_dict[s]}
    for s in S if len(valid_actions_dict[s]) > 0
}

# Iterative Policy Evaluation Sparse
V_eval = iterative_policy_evaluation_sparse(
    pi=pi_uniform,
    S=list(S),
    A=list(set(a for a_list in valid_actions_dict.values() for a in a_list)),
    model=model,
    terminal_states=[],
    valid_actions_dict=valid_actions_dict
)

#  Affichage partiel des valeurs
print("Valeur des états sous politique uniforme (V), premiers états :")
for s in list(V_eval.keys())[:10]:
    print(f"État {s} → V(s) = {V_eval[s]:.4f}")


Valeur des états sous politique uniforme (V), premiers états :
État 0 → V(s) = -446.0098
État 256 → V(s) = -449.6546
État 261 → V(s) = -449.5747
État 272 → V(s) = -454.2522
État 277 → V(s) = -455.3098
État 288 → V(s) = -456.4065
État 293 → V(s) = -455.7246
État 320 → V(s) = -456.8350
État 325 → V(s) = -456.5115
État 512 → V(s) = -453.3581


In [16]:
#  Value Iteration Sparse
pi_vi, V_vi = value_iteration_sparse(
    S=list(S),
    A=list(set(a for a_list in valid_actions_dict.values() for a in a_list)),
    R=R,
    model=model,
    terminal_states=[],  # SecretEnv3 ne définit pas d'états terminaux explicites
    valid_actions_dict=valid_actions_dict
)

# Affichage partiel de la politique
print("Politique extraite (Value Iteration), premiers états connus :")
for s in list(pi_vi.keys())[:10]:
    print(f"État {s} → Politique : {pi_vi[s]}")


Politique extraite (Value Iteration), premiers états connus :
État 0 → Politique : {1: 0.0, 2: 0.0}
État 256 → Politique : {1: 0.0, 2: 0.0}
État 261 → Politique : {0: 0.0, 2: 0.0, 1: 0.0}
État 272 → Politique : {1: 0.0, 2: 0.0}
État 277 → Politique : {2: 0.0, 1: 0.0, 0: 0.0}
État 288 → Politique : {2: 0.0, 1: 0.0}
État 293 → Politique : {2: 0.0, 0: 0.0, 1: 0.0}
État 320 → Politique : {1: 0.0, 2: 0.0}
État 325 → Politique : {0: 0.0, 2: 0.0, 1: 0.0}
État 512 → Politique : {2: 0.0, 1: 0.0}
